In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Multiple Linear Regression From Scratch

In this notebook, we will implement **Multiple Linear Regression from scratch** using NumPy.

We will use the **50 Startups dataset** to predict `Profit` based on:

- R&D Spend
- Administration
- Marketing Spend

We will implement the mathematical formula ourselves instead of using
`sklearn.linear_model.LinearRegression`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 2. Loading the Dataset

We will load the 50 Startups dataset using Pandas and inspect the first few rows.

In [3]:
import os

for  dirname,_,filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname,filename))

/kaggle/input/datasets/amineoumous/50-startups-data/50_Startups.csv


In [6]:
df = pd.read_csv("/kaggle/input/datasets/amineoumous/50-startups-data/50_Startups.csv")
df.head()

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


In [7]:
df.shape
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   R&D Spend        50 non-null     float64
 1   Administration   50 non-null     float64
 2   Marketing Spend  50 non-null     float64
 3   State            50 non-null     object 
 4   Profit           50 non-null     float64
dtypes: float64(4), object(1)
memory usage: 2.1+ KB


,R&D Spend,Administration,Marketing Spend,Profit
count,50.000000,50.000000,50.000000,50.000000
mean,73721.615600,121344.639600,211025.097800,112012.639200
std,45902.256482,28017.802755,122290.310726,40306.180338
min,0.000000,51283.140000,0.000000,14681.400000
25%,39936.370000,103730.875000,129300.132500,90138.902500
50%,73051.080000,122699.795000,212716.240000,107978.190000
75%,101602.800000,144842.180000,299469.085000,139765.977500
max,165349.200000,182645.560000,471784.100000,192261.830000


In [8]:
df.isnull().sum()

R&D Spend          0
Administration     0
Marketing Spend    0
State              0
Profit             0
dtype: int64

## 4. Separating Features and Target

In Multiple Linear Regression, we have:

- **Independent variables (X)** → variables used to make predictions
- **Dependent variable (y)** → the variable we want to predict

For our dataset:

**X:**
- R&D Spend
- Administration
- Marketing Spend

**y:**
- Profit

Our model will learn the relationship:

$$
Profit = b_0 + b_1(R\&D\ Spend) + b_2(Administration) + b_3(Marketing\ Spend)
$$

In [9]:
X = df[['R&D Spend','Administration','Marketing Spend']].values
y = df['Profit'].values

In [10]:
print("X shape:",X.shape)
print("y shape:",y.shape)


X shape: (50, 3)
y shape: (50,)


## 5. Train-Test Split

We divide our dataset into two parts:

- **Training data** → used to learn the coefficients of the model
- **Testing data** → used to evaluate how well the model performs on unseen data

We will use:

- 80% of the data for training
- 20% of the data for testing

Since our dataset contains 50 rows:

$$
80\% = 40\ rows
$$

$$
20\% = 10\ rows
$$

In [11]:
from sklearn.model_selection import train_test_split

In [12]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size = 0.2, random_state = 42
)

In [13]:
print("X_train:",X_train.shape)
print("X_test:",X_test.shape)
print("y_train:",y_train.shape)
print("y_test:",y_test.shape)


X_train: (40, 3)
X_test: (10, 3)
y_train: (40,)
y_test: (10,)


## 6. Adding the Intercept

The equation for Multiple Linear Regression is:

$$
y = b_0 + b_1x_1 + b_2x_2 + b_3x_3
$$

Here:

- $b_0$ → Intercept
- $b_1, b_2, b_3$ → Coefficients
- $x_1, x_2, x_3$ → Input features

To calculate all coefficients using the matrix formula, we add a column
of **1s** to our feature matrix.

The new matrix becomes:

$$
X =
\begin{bmatrix}
1 & x_1 & x_2 & x_3 \\
1 & x_1 & x_2 & x_3 \\
1 & x_1 & x_2 & x_3
\end{bmatrix}
$$

In [20]:
X_train = np.c_[np.ones(X_train.shape[0]),X_train]
X_test = np.c_[np.ones(X_test.shape[0]),X_test]

In [27]:
X_train.shape

(40, 10)

In [29]:
X_train[:5]

array([[1.0000000e+00, 1.0000000e+00, 1.0000000e+00, 1.0000000e+00,
        1.0000000e+00, 1.0000000e+00, 1.0000000e+00, 9.3863750e+04,
        1.2732038e+05, 2.4983944e+05],
       [1.0000000e+00, 1.0000000e+00, 1.0000000e+00, 1.0000000e+00,
        1.0000000e+00, 1.0000000e+00, 1.0000000e+00, 1.4210734e+05,
        9.1391770e+04, 3.6616842e+05],
       [1.0000000e+00, 1.0000000e+00, 1.0000000e+00, 1.0000000e+00,
        1.0000000e+00, 1.0000000e+00, 1.0000000e+00, 4.4069950e+04,
        5.1283140e+04, 1.9702942e+05],
       [1.0000000e+00, 1.0000000e+00, 1.0000000e+00, 1.0000000e+00,
        1.0000000e+00, 1.0000000e+00, 1.0000000e+00, 1.2054252e+05,
        1.4871895e+05, 3.1161329e+05],
       [1.0000000e+00, 1.0000000e+00, 1.0000000e+00, 1.0000000e+00,
        1.0000000e+00, 1.0000000e+00, 1.0000000e+00, 1.4437241e+05,
        1.1867185e+05, 3.8319962e+05]])

## 7. Multiple Linear Regression From Scratch

We will calculate the regression coefficients using the **Normal Equation**:

$$
\beta = (X^TX)^{-1}X^Ty
$$

Where:

- $X^T$ → Transpose of X
- $(X^TX)^{-1}$ → Inverse of $X^TX$
- $y$ → Target values
- $\beta$ → Coefficients of the model

The coefficients will contain:

$$
\beta =
\begin{bmatrix}
b_0 \\
b_1 \\
b_2 \\
b_3
\end{bmatrix}
$$

In [39]:
class MultipleLinearRegression:

    def __init__(self):
        self.coefficient = None
    def fit(self,X,y):
        X_T = X.T
        XTX = np.dot(X_T,X)
        XTX_inverse = np.linalg.pinv(XTX)
        XTy = np.dot(X_T,y)
        self.coefficients = np.dot(XTX_inverse,XTy)
    def predict(self,X):
        return np.dot(X,self.coefficients)

In [40]:
model = MultipleLinearRegression()

In [41]:
model.fit(X_train,y_train)

In [42]:
model.coefficients

array([ 7.72455526e+03,  7.72454886e+03,  7.72455613e+03,  7.72455526e+03,
        7.72455526e+03,  7.72455526e+03,  7.72455526e+03,  8.03779286e-01,
       -6.79292104e-02,  3.12415384e-02])

## 8. Making Predictions

After training, our model has learned the coefficients:

$$
\beta =
\begin{bmatrix}
b_0 \\
b_1 \\
b_2 \\
b_3
\end{bmatrix}
$$

To predict Profit, we use:

$$
\hat{y} = X\beta
$$

where:

- $X$ → input features
- $\beta$ → learned coefficients
- $\hat{y}$ → predicted Profit

In [43]:
y_pred = model.predict(X_test)

In [44]:
comparison = pd.DataFrame({"Actual Profit :": y_test,
                          "Predicted Profit": y_pred})
comparison

,Actual Profit :,Predicted Profit
0,134307.35,126703.026848
1,81005.76,84894.752438
2,99937.59,98893.419463
3,64926.08,46501.709182
4,125370.37,129128.396482
5,35673.41,50992.698538
6,105733.54,109016.553741
7,107404.34,100878.464256
8,97427.84,97700.597547
9,122776.86,113106.153781


## 9. Model Evaluation — Mean Squared Error

Mean Squared Error (MSE) measures the average squared difference between
the actual values and predicted values.

$$
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
$$

Where:

- $y_i$ = actual value
- $\hat{y}_i$ = predicted value
- $n$ = number of observations

A **lower MSE** means the predictions are closer to the actual values.

In [45]:
mse = np.mean((y_test-y_pred)**2)
print("mse:",mse)

mse: 80926327.81925607
